# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/ppo/#ppo_ataripy

In [2]:
# @launchit.collected

In [3]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import datetime
import json
import pprint
import re
import uuid
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm
from tqdm import tqdm as tqdm_text

import numpy as np
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions.categorical import Categorical
import torch.multiprocessing as torch_mp # not actually needed but keeped to be aligned with docs (see https://github.com/pytorch/pytorch/issues/70041)

import gymnasium as gym
import ale_py
import av

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect

from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter
from logging_utils import *
from artifact_registry import *
from torch_utils import *
import launchit
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

# Init

In [4]:
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()
    
def create_config():
    config = namedtuple('Config', 
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                        'self_fname, self_name, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, exec_mode, is_interactive')(
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        self_fname=None,
        self_name=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf)['jupyter_session']
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
            config = config._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)
    
    config = config._replace(is_interactive=config.exec_mode != ExecMode.LAUNCH_MODULE)    
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    return config

In [5]:
# @launchit.disable_2
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17a_ppo_atari_mp_03.ipynb',
 'self_name': '17a_ppo_atari_mp_03',
 'subproject_name': '17_rl',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [6]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()
    WORKER = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    # System params
    random_seed: int = 1
    torch_deterministic: bool = True 

    # Environment params
    env_id: str = None

    # Agent params
    separate_networks: bool = False # where Actor / Critic have separate networks or share share the same network

    # Worker params
    workers_count: int = 8 # number of parallel game environments, each is run by separate worker
    
    # Training procedure params (PPO related) 
    global_steps_count: int = 10_000_000 # total number of steps 
    rollout_steps_count: int = 128 # how many steps to run in a single policy rolllout
    minibatches_count: int = 4
    epochs_count: int = 4 
    clip_coef: float = 0.1 # the surrogate clipping coefficient
    clip_vloss: bool = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    ent_coef: float = 0.01 # coefficient of the entropy
    vf_coef: float = 0.5 # coefficient of the value function
    max_grad_norm: float = 0.5 # the maximum norm for the gradient clipping
    target_kl: float = None # e target KL divergence threshold
    norm_adv: bool = True # Toggles advantages normalization
    
    # RL params
    gamma: float = 0.99 # the discount factor gamma
    gae_lambda: float = 0.95 # the lambda for the general advantage estimation

    # Video params
    capture_video: str = 'every(1000000)' # video capture policy depending on steps

    # Optimization params
    optimizer: str = 'Adam(eps=0.00001)'
    learn_rate: str = '0.00025,linear()'
    
    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()
HP.random_seed = 42

# Launch

## LaunchState

In [7]:
@dataclass(slots=True)
class LaunchState:
    mp_ctx: object = None
    env: object = None # in master mode used only as provider of configuration parameters
    workers: list = None 
    capture_video_worker: object = None 
    agent: object = None

## new_artifact_registry

In [8]:
# @launchit.disable_2
def new_artifact_registry(is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ArtifactRegistry(CONFIG.model_group_uri)

## new_summary_writer

In [9]:
# @launchit.disable_2
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Create

In [10]:
# @launchit.disable_2
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', HP.launch_id)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    random.seed(HP.random_seed)
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if HP.torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()
artifact_registry = new_artifact_registry()
artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    
meta = dict(
    optuna_trial_number=getattr(optuna_trial, 'number', None),
    hypers=HP._asdict(), 
    config=CONFIG._asdict(), 
)

with io.StringIO() as b:
    json.dump(meta, b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = lc.name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

LS = LaunchState()
LS.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch

Random seed=42
torch.backends.cudnn.deterministic=True
Tensorboard run=17a_ppo_atari_mp_03/0


# Environment

## create_env

In [11]:
def create_env(env_id, video_dir_name=None):
    if video_dir_name is not None:
        env = gym.make(env_id, render_mode='rgb_array')
        env = gym.wrappers.RecordVideo(env, video_dir_name, episode_trigger=lambda episode_id: episode_id == 0)
    else:
        env = gym.make(env_id)

    env = gym.wrappers.RecordEpisodeStatistics(env)
    env = NoopResetEnv(env, noop_max=30)
    env = MaxAndSkipEnv(env, skip=4)
    env = EpisodicLifeEnv(env) 
    
    if 'FIRE' in env.unwrapped.get_action_meanings():
        env = FireResetEnv(env)
        
    env = gym.wrappers.Autoreset(env)
    env = ClipRewardEnv(env)
    # Following three directly influence env.observation_space
    env = gym.wrappers.ResizeObservation(env, (84, 84))
    env = gym.wrappers.GrayscaleObservation(env)
    env = gym.wrappers.FrameStackObservation(env, 4)
    return env

## Configure 

In [12]:
# @launchit.disable
# @launchit.collect_1
HP.env_id = "BreakoutNoFrameskip-v4"
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': False,
 'workers_count': 8,
 'global_steps_count': 10000000,
 'rollout_steps_count': 128,
 'minibatches_count': 4,
 'epochs_count': 4,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'capture_video': 'every(1000000)',
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()'}


## Create

In [13]:
# @launchit.disable_2
LS.env = create_env(HP.env_id)
assert isinstance(LS.env.action_space, gym.spaces.Discrete), "only discrete action space is supported"
LOG(f'{LS.env.observation_space=}, {LS.env.action_space=}, {LS.env.metadata=}')

LS.env.observation_space=Box(0, 255, (4, 84, 84), uint8), LS.env.action_space=Discrete(4), LS.env.metadata={'render_modes': ['human', 'rgb_array'], 'render_fps': 30}


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


# Agent

## Agent

In [14]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = 512
        actions_count: int = None 
        separate_networks: bool = None
    
    def __init__(self, params):
        super().__init__()
        self.params = params

        if params.separate_networks:
            self.actor = nn.Sequential(
                self._layer_init(nn.Conv2d(4, 32, 8, stride=4)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                self._layer_init(nn.Linear(64 * 7 * 7, params.d_model)),
                nn.ReLU(),
                self._layer_init(nn.Linear(params.d_model, params.actions_count), std=0.01)
            )

            self.critic = nn.Sequential(
                self._layer_init(nn.Conv2d(4, 32, 8, stride=4)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                self._layer_init(nn.Linear(64 * 7 * 7, params.d_model)),
                nn.ReLU(),
                self._layer_init(nn.Linear(params.d_model, 1), std=1)
            )
        else:
            self.network = nn.Sequential(
                self._layer_init(nn.Conv2d(4, 32, 8, stride=4)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                self._layer_init(nn.Linear(64 * 7 * 7, params.d_model)),
                nn.ReLU(),
            )
            self.actor = self._layer_init(nn.Linear(params.d_model, params.actions_count), std=0.01)
            self.critic = self._layer_init(nn.Linear(params.d_model, 1), std=1)

    def get_value(self, x):
        if self.params.separate_networks:
            return self.critic(x / 255.0)
        else:
            hidden = self.network(x / 255.0)
            return self.critic(hidden)

    ForwardResult = namedtuple('ForwardResult', 'action, log_action_probs, probs_entropy, value')

    def forward(self, x, action=None): # get_action_and_value
        if self.params.separate_networks:
            x = x / 255.0
            logits = self.actor(x) # [batch, actions] 
            probs = Categorical(logits=logits) # [batch]
            action = lu.coalesce(action, lambda: probs.sample()) # [batch]
            return Agent.ForwardResult(
                action=action, 
                log_action_probs=probs.log_prob(action), 
                probs_entropy=probs.entropy(), 
                value=self.critic(x), # [batch, 1]
            )
        else:
            hidden = self.network(x / 255.0) # [batch, d_model] 
            logits = self.actor(hidden) # [batch, actions] 
            probs = Categorical(logits=logits) # [batch]
            action = lu.coalesce(action, lambda: probs.sample()) # [batch]
            return Agent.ForwardResult(
                action=action, 
                log_action_probs=probs.log_prob(action), 
                probs_entropy=probs.entropy(), 
                value=self.critic(hidden), # [batch, 1]
            )

    @staticmethod
    def _layer_init(layer, std=np.sqrt(2), bias_const=0.0):
        torch.nn.init.orthogonal_(layer.weight, std)
        torch.nn.init.constant_(layer.bias, bias_const)
        return layer

## Smoke test

In [15]:
# @launchit.disable
ap = Agent.Params(
    actions_count=10,
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
print(agent)
params_count = sum(p.numel() for p in agent.parameters())
print(f'{params_count=:_}')
probe_batch = torch.zeros((2, 4, 84, 84)).to(CONFIG.cuda_device)
print(f'{probe_batch.shape=}')
r = agent(probe_batch)
print(f'{r.action.shape=}, {r.log_action_probs.shape=}, {r.probs_entropy.shape=}, {r.value.shape=}')

Agent(
  (network): Sequential(
    (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=512, bias=True)
    (8): ReLU()
  )
  (actor): Linear(in_features=512, out_features=10, bias=True)
  (critic): Linear(in_features=512, out_features=1, bias=True)
)
params_count=1_689_771
probe_batch.shape=torch.Size([2, 4, 84, 84])
r.action.shape=torch.Size([2]), r.log_action_probs.shape=torch.Size([2]), r.probs_entropy.shape=torch.Size([2]), r.value.shape=torch.Size([2, 1])


## Quick play test

In [16]:
# @launchit.disable
ap = Agent.Params(
    actions_count=LS.env.action_space.n.item(),
)
agent = Agent(ap)
agent = agent.to(CONFIG.cuda_device)
env = create_env(HP.env_id)
obs, _ = env.reset(seed=HP.random_seed)
device = next(iter(agent.parameters())).device

with eval_guard(agent):
    with torch.no_grad():
        for step in tqdm(range(0, 1000)): 
            obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
            action = agent(obs).action.cpu().numpy()[0]
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                if env.get_wrapper_attr('was_real_done'):
                    assert 'episode' in info
                    
                    if 'episode' in info: # 'episode' is a default stats_key for RecordEpisodeStatistics
                        episode_stats = info['episode']
                        print(f'{step:05}', episode_stats)
                else:
                    assert not 'episode' in info
            else:
                assert not 'episode' in info

  0%|          | 0/1000 [00:00<?, ?it/s]

00191 {'r': 2.0, 'l': 816, 't': 0.817427}
00309 {'r': 0.0, 'l': 534, 't': 0.554276}
00427 {'r': 0.0, 'l': 523, 't': 0.463597}
00573 {'r': 1.0, 'l': 639, 't': 0.615195}
00816 {'r': 3.0, 'l': 1006, 't': 0.985125}


## Configure

In [17]:
# @launchit.disable
# @launchit.collect_1
HP.separate_networks = False
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': False,
 'workers_count': 8,
 'global_steps_count': 10000000,
 'rollout_steps_count': 128,
 'minibatches_count': 4,
 'epochs_count': 4,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'capture_video': 'every(1000000)',
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()'}


## Create

In [18]:
# @launchit.disable_2
ap = Agent.Params(
    actions_count=LS.env.action_space.n.item()
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)

# Video

## get_fresh_video_dir_name

In [19]:
def get_fresh_video_dir_name():
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S") # generate unique dir name in order to shut up RecordVideo from complaining
    return os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}')

## capture_video_of_test_rollout

In [20]:
def capture_video_of_test_rollout(agent, max_steps_count=10_000, video_dir_name=None):
    video_dir_name = lu.coalesce(video_dir_name, lambda: get_fresh_video_dir_name())
    assert video_dir_name is not None
    env = create_env(HP.env_id, video_dir_name)
    assert env.action_space.n.item() == agent.params.actions_count
    obs, _ = env.reset(seed=HP.random_seed)
    device = next(iter(agent.parameters())).device
    
    with eval_guard(agent):
        with torch.no_grad():
            for step in range(0, max_steps_count): 
                obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(device)
                action = agent(obs).action.cpu().numpy()[0]
                obs, reward, terminated, truncated, info = env.step(action)
                
                if (terminated or truncated) and env.get_wrapper_attr('was_real_done'):
                    break

    game_meta = dict(
        reward=env.get_wrapper_attr('episode_returns'),
        frames_count=env.get_wrapper_attr('episode_lengths'),
        steps_count=step + 1,
    )
    
    env.close() # this forces video recording to complete and write video file
    
    video_fnames = list(filter(lambda fn: os.path.isfile(os.path.join(video_dir_name, fn)), os.listdir(video_dir_name)))
    assert len(video_fnames) == 1, len(video_fnames)
    video_fname = os.path.join(video_dir_name, video_fnames[0])
    video_meta = {}
    
    with open(video_fname, 'rb') as f:
        container = av.open(f)
        video_meta['fps'] = float(container.streams.video[0].average_rate)
        video_stream = container.streams.video[0]
        video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
        
    return video_fname, dict(game=game_meta, video=video_meta)

In [21]:
# @launchit.disable
capture_video_of_test_rollout(LS.agent)

('/home/misha/dev/mine/neurolab/run/17_rl/video-17a_ppo_atari_mp_03-launch0-2026.05.04-21:41:57/rl-video-episode-0.mp4',
 {'game': {'reward': 1.0, 'frames_count': 624, 'steps_count': 144},
  'video': {'fps': 30.0, 'duration': 20.833333333333332}})

# Worker

Implementation details regarding memory sharing between PyTorch applications: <a href="./dialogs/torch-multiprocessing-shm.ipynb">torch-multiprocessing-shm.ipynb</a>

## WorkerTask

In [22]:
# Exchange data between main and child processes
@dataclass(slots=True)
class WorkerTask:
    task_id: int
    op: str
    params: dict = None

@dataclass(slots=True)
class WorkerTaskResult:
    task_id: int
    payload: object = None

## WorkerCtl

In [23]:
# Master's stuff (main process)
class WorkerCtl:
    task_id = 0
    
    def __init__(self, worker_ind, module, mp_ctx):
        self.task_ctor = getattr(module, 'WorkerTask')
        self.worker_ind = worker_ind
        self.task_queue = mp_ctx.Queue()
        self.result_queue = mp_ctx.Queue()
        self.process = mp_ctx.Process(target=getattr(module, 'worker_loop'), args=(worker_ind, self.task_queue, self.result_queue))
        self.process.start()
        self.pending_task_ids = deque()

    @staticmethod
    def gen_task_id():
        WorkerCtl.task_id += 1
        return WorkerCtl.task_id

    def healthcheck(self):
        task = self.task_ctor(task_id=self.gen_task_id(), op='HEALTHCHECK')
        self.task_queue.put(task)
        self.result_queue.get()
        
    def terminate(self, timeout=None):
        if self.process.is_alive():
            task = self.task_ctor(task_id=self.gen_task_id(), op='TERMINATE')
            try:
                self.task_queue.put(task)
                self.result_queue.get(timeout=timeout)
                self.process.join()
            except:
                self.process.terminate()

    def init_agent(self, agent_params, agent_state_dict=None, device=None):
        if agent_state_dict is not None:
            for key in agent_state_dict:
                assert agent_state_dict[key].is_shared()
        
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='INIT_AGENT', 
            params=dict(
                agent_params=dataclasses.asdict(agent_params),
                agent_state_dict=agent_state_dict,
                device=device,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def sync_agent(self, agent_state_dict):
        for key in agent_state_dict:
            assert agent_state_dict[key].is_shared()
                
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='SYNC_AGENT',
            params=dict(agent_state_dict=agent_state_dict),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def init_rollout(self):
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='INIT_ROLLOUT', 
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id
        
    def rollout(self, steps_count, obs, actions, logprobs, rewards, dones, values, last_episode_rewards, last_episode_lengths, next_obs, next_done):
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='ROLLOUT', 
            params=dict(
                steps_count=steps_count,
                # Tensors below are subject to fast GPU-TO-GPU transfer
                obs=obs,
                actions=actions,
                logprobs=logprobs,
                rewards=rewards,
                dones=dones,
                values=values,
                last_episode_rewards=last_episode_rewards,
                last_episode_lengths=last_episode_lengths,
                next_obs=next_obs,
                next_done=next_done,
            ), 
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id
    
    def capture_video_of_test_rollout(self, max_steps_count, video_dir_name, forward_data=None):
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='CAPTURE_VIDEO',
            params=dict(
                max_steps_count=max_steps_count, 
                video_dir_name=video_dir_name,
                forward_data=forward_data,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def is_busy(self):
        return len(self.pending_task_ids) > 0
    
    def get_task_result(self):
        assert len(self.pending_task_ids) > 0
        result = self.result_queue.get()
        assert result.task_id == self.pending_task_ids.popleft()
        return result

    def peek_task_result(self):
        if not self.pending_task_ids:
            return None
        
        try:
            result = self.result_queue.get(block=False)
            assert result.task_id == self.pending_task_ids.popleft()
            return result
        except queue.Empty:
            return None # task is not completed yet

    def drain_task_results(self):
        results = []

        while self.pending_task_ids:
            task_id = self.pending_task_ids.popleft()
            task_result = self.result_queue.get()
            assert task_result.task_id == task_id
            results.append(task_result)

        return results

## worker_loop

In [24]:
# Executed in child process
def worker_loop(worker_ind, inp_queue, out_queue):
    @dataclass(slots=True)
    class WorkerState:
        agent: object = None
        is_attached_agent: bool = False
        env: object = None
    
    CONFIG = create_config()
    LOG = Logging.get()
    LOG.app_name = CONFIG.self_name
    LOG.enable('syslog', True)
    LOG.enable('stdout', False)
    worker_random_seed = HP.random_seed + worker_ind
    
    with LOG.auto_prefix('WRK', worker_ind, 'SEED', worker_random_seed):
        LOG(f'CONFIG={CONFIG._asdict()}')
        
        au.init()
        random.seed(worker_random_seed)
        torch.manual_seed(worker_random_seed)
        RNG = np.random.default_rng(worker_random_seed)
        LOG(f'{worker_random_seed=}')
        
        torch.backends.cudnn.deterministic = HP.torch_deterministic
        LOG(f'{torch.backends.cudnn.deterministic=}')

        WS = WorkerState()
        LOG('Worker is ready')
        
        task_wait_timeout = 60
        is_running = True
    
        while is_running:
            try:
                # task is expected to be an instanace of WorkerTask class
                task = inp_queue.get(block=True, timeout=task_wait_timeout)
            except queue.Empty:
                LOG(f'Didn\'t get any tasks within {task_wait_timeout} seconds, waiting again')
                continue

            with LOG.auto_prefix('TASK', task.task_id):
                LOG(f'Got task #{task.task_id} {task.op}')
                task_result = WorkerTaskResult(task_id=task.task_id)
                
                match task.op:
                    case 'HEALTHCHECK':
                        pass
                    case 'TERMINATE':
                        is_running = False
                    case 'INIT_AGENT':
                        ap = Agent.Params(**task.params['agent_params'])
                        device = lu.coalesce(task.params.get('device'), CONFIG.cuda_device)
                        WS.agent = Agent(ap).to(device)
                        WS.is_attached_agent = task.params['agent_state_dict'] is not None
                        
                        if WS.is_attached_agent:
                            # Storage for weights of attached agent is pointed to agent's weights in main process
                            state_dict = task.params['agent_state_dict']
                            
                            with torch.no_grad():
                                for name, param in WS.agent.named_parameters():
                                    assert state_dict[name].is_shared()
                                    param.data = state_dict[name]

                        LOG(f'{WS.is_attached_agent=}')
                    case 'SYNC_AGENT':
                        assert WS.agent is not None
                        assert not WS.is_attached_agent
                        
                        state_dict = task.params['agent_state_dict']
                        
                        with torch.no_grad():
                            # Fast GPU-TO-GPU sync
                            for name, param in WS.agent.named_parameters():
                                assert state_dict[name].is_shared()
                                param.copy_(state_dict[name])
                    case 'INIT_ROLLOUT':
                        WS.env = create_env(HP.env_id)
                        assert isinstance(WS.env.action_space, gym.spaces.Discrete), "only discrete action space is supported"
                        LOG(f'Env created: {WS.env.observation_space=}, {WS.env.action_space=}, {WS.env.metadata=}')

                        next_obs, _ = WS.env.reset(seed=worker_random_seed)
                        next_obs = torch.Tensor(next_obs).unsqueeze(0).to(CONFIG.cuda_device)
                        next_done = False
                        LOG('Env reset')
                    case 'ROLLOUT':
                        assert WS.agent is not None
                        assert WS.env is not None
                        assert WS.env.action_space.n.item() == WS.agent.params.actions_count, (WS.env.action_space.n.item(), WS.agent.params.actions_count)
                        
                        steps_count = task.params['steps_count']
                        obs, dones, values, actions, logprobs, rewards, last_episode_rewards, last_episode_lengths = (
                            task.params['obs'],
                            task.params['dones'],
                            task.params['values'],
                            task.params['actions'],
                            task.params['logprobs'],
                            task.params['rewards'],
                            task.params['last_episode_rewards'],
                            task.params['last_episode_lengths'],
                        )

                        my_actions = np.zeros((steps_count,) + WS.env.action_space.shape, dtype=np.float32)
                        my_logprobs = np.zeros(steps_count, dtype=np.float32)
                        my_values = np.zeros(steps_count, dtype=np.float32)
                        my_rewards = np.zeros(steps_count, dtype=np.float32)
                        my_dones = np.zeros(steps_count, dtype=np.float32)
                        
                        with torch.no_grad():
                            for step in range(steps_count): 
                                obs[step,worker_ind] = next_obs
                                my_dones[step] = next_done
        
                                action, logprob, _, value = WS.agent(next_obs)
                                action = action.item()
                                my_actions[step] = action
                                my_logprobs[step] = logprob.item()
                                my_values[step] = value.item()
        
                                next_obs, reward, termination, truncation, info = WS.env.step(action)
                                next_obs = torch.Tensor(next_obs).unsqueeze(0)
                                next_obs = next_obs.to(CONFIG.cuda_device, non_blocking=True) # heavy thing, non_blocking may speed up a little
                                next_done = termination or truncation
                                my_rewards[step] = reward
                        
                                if 'episode' in info: # 'episode' is a default stats_key for RecordEpisodeStatistics
                                    episode_stats = info['episode']
                                    last_episode_rewards[worker_ind] = episode_stats['r']
                                    last_episode_lengths[worker_ind] = episode_stats['l']

                        actions[:,worker_ind] = torch.tensor(my_actions).to(CONFIG.cuda_device, non_blocking=True)
                        logprobs[:,worker_ind] = torch.tensor(my_logprobs).to(CONFIG.cuda_device, non_blocking=True)
                        values[:,worker_ind] = torch.tensor(my_values).to(CONFIG.cuda_device, non_blocking=True)
                        rewards[:,worker_ind] = torch.tensor(my_rewards).to(CONFIG.cuda_device, non_blocking=True)
                        dones[:,worker_ind] = torch.tensor(my_dones).to(CONFIG.cuda_device, non_blocking=True)
                        task.params['next_obs'][worker_ind] = next_obs 
                        task.params['next_done'][worker_ind] = next_done

                        LOG(f'Done rollout for {steps_count} steps')
                    case 'CAPTURE_VIDEO':
                        assert WS.agent is not None
                        
                        with torch.no_grad():
                            video_fname, video_meta = capture_video_of_test_rollout(
                                WS.agent, 
                                max_steps_count=task.params['max_steps_count'], 
                                video_dir_name=task.params['video_dir_name'])
                            task_result.payload = (video_fname, video_meta, task.params['forward_data'])
                    case _:
                        LOG(f'Unknown {task.op=}, ignoring')
        
                out_queue.put(task_result)
                LOG('Task complete')
        
        LOG('Worker is going down')

## get_worker_factory

In [24]:
def get_worker_factory():
    with LOG.auto_log_level(logging.INFO):
        expandvars = dict(
            PROJECT_ROOT_PATH=CONFIG.project_root_path,
            MODEL_NAME=CONFIG.self_name,
            MODEL_VERSION=HP.launch_component().version,
            LAUNCH_GOAL=LaunchGoal.WORKER.value,
        )
        module_fname = launchit.launchit(
            CONFIG.self_fname, 
            expandvars=expandvars, 
            make_py_file=True,
            dir_name=CONFIG.run_path,
            disable_inds=[2]
        )
        LOG.info(f'Created "{module_fname}"')
        
    module_dir_name = os.path.dirname(module_fname)
    module_name = os.path.splitext(os.path.basename(module_fname))[0]
    sys.path.append(module_dir_name)
    module = __import__(module_name)

    def factory(worker_ind):
        return WorkerCtl(worker_ind, module, LS.mp_ctx)

    return factory

## Test

### rollout

In [25]:
# @launchit.disable
test_workers_count = 4

wf = get_worker_factory()
test_workers = [wf(i) for i in range(test_workers_count)]

try:
    for w in test_workers: 
        w.init_agent(agent_params=LS.agent.params, agent_state_dict=LS.agent.state_dict())
        w.init_rollout()
        w.drain_task_results()
    
    test_global_steps_count = 5_000
    test_rollout_steps_count = 100
    
    test_next_obs = torch.zeros((len(test_workers),) + LS.env.observation_space.shape).to(CONFIG.cuda_device)
    test_next_done = torch.zeros(len(test_workers)).to(CONFIG.cuda_device)
    test_obs = torch.zeros((test_rollout_steps_count, len(test_workers)) + LS.env.observation_space.shape).to(CONFIG.cuda_device)
    test_actions = torch.zeros((test_rollout_steps_count, len(test_workers)) + LS.env.action_space.shape).to(CONFIG.cuda_device)
    test_logprobs = torch.zeros((test_rollout_steps_count, len(test_workers))).to(CONFIG.cuda_device)
    test_rewards = torch.zeros((test_rollout_steps_count, len(test_workers))).to(CONFIG.cuda_device)
    test_dones = torch.zeros((test_rollout_steps_count, len(test_workers))).to(CONFIG.cuda_device)
    test_values = torch.zeros((test_rollout_steps_count, len(test_workers))).to(CONFIG.cuda_device)
    test_last_episode_rewards = torch.zeros(len(test_workers))
    test_last_episode_lengths = torch.zeros(len(test_workers))
    test_last_episode_rewards.share_memory_()
    test_last_episode_lengths.share_memory_()

    test_step = 0
    
    with tqdm(total=test_global_steps_count) as pbar:
        while test_step < test_global_steps_count:
            for w in test_workers:
                w.rollout(test_rollout_steps_count, 
                          test_obs, test_actions, test_logprobs, test_rewards, test_dones, test_values, 
                          test_last_episode_rewards, test_last_episode_lengths, 
                          test_next_obs, test_next_done)
        
            for w in test_workers:
                w.get_task_result()

            step_inc = test_rollout_steps_count * len(test_workers)
            test_step += step_inc
            pbar.update(step_inc)
finally:
    for w in test_workers: 
        w.terminate(timeout=3)

Created "/home/misha/dev/mine/neurolab/run/17_rl/17a_ppo_atari_mp_03-launch13.py"


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


  0%|          | 0/5000 [00:00<?, ?it/s]

### capture_video_of_test_rollout

In [26]:
# @launchit.disable
wf = get_worker_factory()
test_worker = wf(0)

try:
    test_worker.init_agent(LS.agent.params)
    test_worker.get_task_result()
    
    video_dir_name = get_fresh_video_dir_name()
    tid = test_worker.sync_agent(LS.agent.state_dict())
    test_worker.capture_video_of_test_rollout(max_steps_count=2000, video_dir_name=video_dir_name)
    tr = test_worker.get_task_result() # wait for weights are synced is done
    assert tr.task_id == tid
    result = test_worker.drain_task_results()[-1]
    print(result)
finally:
    test_worker.terminate()

Created "/home/misha/dev/mine/neurolab/run/17_rl/17a_ppo_atari_mp_03-launch14.py"


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


WorkerTaskResult(task_id=67, payload=('/home/misha/dev/mine/neurolab/run/17_rl/video-17a_ppo_atari_mp_03-launch0-2026.05.04-20:10:30/rl-video-episode-0.mp4', {'game': {'reward': 1.0, 'frames_count': 622, 'steps_count': 143}, 'video': {'fps': 30.0, 'duration': 20.766666666666666}}, None))


## Configure

In [27]:
# @launchit.disable
# @launchit.collect_1
HP.workers_count = 4
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': False,
 'workers_count': 4,
 'global_steps_count': 10000000,
 'rollout_steps_count': 128,
 'minibatches_count': 4,
 'epochs_count': 4,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'capture_video': 'every(1000000)',
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()'}


## Create

In [28]:
# @launchit.disable_2
wf = get_worker_factory()
LS.workers = [wf(i) for i in range(HP.workers_count)]

# Init in serial, shows to be much faster than in parallel (GPU contention issues?)
for w in LS.workers: 
    w.init_agent(agent_params=LS.agent.params, agent_state_dict=LS.agent.state_dict())
    w.init_rollout()
    w.drain_task_results()

LS.capture_video_worker = wf(0)
LS.capture_video_worker.init_agent(agent_params=LS.agent.params)
LS.capture_video_worker.get_task_result()

Created "/home/misha/dev/mine/neurolab/run/17_rl/17a_ppo_atari_mp_03-launch15.py"


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


WorkerTaskResult(task_id=77, payload=None)

# TRAIN

## CaptureVideoManager

In [33]:
class CaptureVideoManager:
    def __init__(self, capture_video_policy):
        ump = hp_parse_universal_module(capture_video_policy)
        self.should_capture_video = self.create_should_capture_video(ump.module_name, *ump.args, **ump.kwargs)
    
    def schedule_capture_video(self, global_step):
        task_id = LS.capture_video_worker.sync_agent(LS.agent.state_dict())
        LS.capture_video_worker.capture_video_of_test_rollout(
            max_steps_count=5_000, 
            video_dir_name=get_fresh_video_dir_name(), 
            forward_data=dict(global_step=global_step)
        )
        task_result = LS.capture_video_worker.get_task_result() # wait until sync weights is finished so we capture video on agent with weights as LS.agent
        assert task_id == task_result.task_id

    def upload_captured_video(self, is_drain=False):
        if is_drain:
            trs = LS.capture_video_worker.drain_task_results()
        else:
            trs = [LS.capture_video_worker.peek_task_result()]

        for tr in filter(lambda tr: tr is not None, trs):
            video_fname, video_meta, forward_data = tr.payload
            _, video_fname_ext = os.path.splitext(video_fname)
            ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
            remote_video_fname = f'{ts}-{forward_data['global_step']:09}.{video_fname_ext.lstrip('.')}'
            summary_writer.add_file(video_fname, remote_video_fname)
            summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
            ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
            summary_writer.add_text('videos', ref_text, forward_data['global_step'])

    @staticmethod
    def create_should_capture_video(policy_name, period):
        if policy_name == 'every':
            last_count = 0
            
            def every_thunk(count):
                nonlocal last_count
                elapsed = count - last_count

                if count == 0 or elapsed >= period:
                    last_count = count
                    return True

                return False

            return every_thunk
        
        assert False, f'Unsupported {policy_name=}'

## Configure

In [26]:
# @launchit.disable
# @launchit.collect

# Training procedure params (PPO related) 
HP.global_steps_count = 10_000 # total number of steps 
HP.rollout_steps_count = 128 # how many steps to run in a single policy rolllout
HP.minibatches_count = 4
HP.epochs_count = 4 
HP.clip_coef = 0.1 # the surrogate clipping coefficient
HP.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
HP.ent_coef = 0.01 # coefficient of the entropy
HP.vf_coef = 0.5 # coefficient of the value function
HP.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
HP.target_kl = None # e target KL divergence threshold
HP.norm_adv = True # Toggles advantages normalization

# RL params
HP.gamma = 0.99 # the discount factor gamma
HP.gae_lambda = 0.95 # the lambda for the general advantage estimation

# Video params
HP.capture_video = 'every(100000)' # video capture policy depending on steps

# Optimization params
HP.optimizer = 'Adam(eps=0.00001)'
HP.learn_rate = '0.00025,linear()'

# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': False,
 'workers_count': 8,
 'global_steps_count': 10000,
 'rollout_steps_count': 128,
 'minibatches_count': 4,
 'epochs_count': 4,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'capture_video': 'every(100000)',
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()'}


## Create

In [34]:
# @launchit.disable_2
batch_size = int(HP.workers_count * HP.rollout_steps_count)
minibatch_data_loader = DataLoader(np.arange(batch_size), shuffle=True, batch_size=batch_size // HP.minibatches_count)

ump = hp_parse_universal_module(HP.optimizer)
assert not ump.args
lr_params = hp_parse_learn_rate(HP.learn_rate)
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), lr=lr_params.learn_rate, **ump.kwargs)
lr_scheduler = LrSchedulerWrapper(optimizer, lr_params, HP.global_steps_count // batch_size)

# A-la scoreboard for current play round (run rollout_steps_count within each of envs)
obs = torch.zeros((HP.rollout_steps_count, HP.workers_count) + LS.env.observation_space.shape).to(CONFIG.cuda_device)
actions = torch.zeros((HP.rollout_steps_count, HP.workers_count) + LS.env.action_space.shape).to(CONFIG.cuda_device)
logprobs = torch.zeros((HP.rollout_steps_count, HP.workers_count)).to(CONFIG.cuda_device)
rewards = torch.zeros((HP.rollout_steps_count, HP.workers_count)).to(CONFIG.cuda_device)
dones = torch.zeros((HP.rollout_steps_count, HP.workers_count)).to(CONFIG.cuda_device)
values = torch.zeros((HP.rollout_steps_count, HP.workers_count)).to(CONFIG.cuda_device)
last_episode_rewards = torch.zeros(HP.workers_count)
last_episode_lengths = torch.zeros(HP.workers_count)
last_episode_rewards.share_memory_()
last_episode_lengths.share_memory_()
next_obs = torch.zeros((HP.workers_count,) + LS.env.observation_space.shape).to(CONFIG.cuda_device)
next_done = torch.zeros(HP.workers_count).to(CONFIG.cuda_device)

for name in ('obs', 'actions', 'logprobs', 'rewards', 'dones', 'values', 'last_episode_rewards', 'last_episode_lengths', 'next_obs', 'next_done'):
    t = globals()[name]
    assert t.is_shared()
    print(f'{name}.shape={t.shape}')

capture_video_manager = CaptureVideoManager(HP.capture_video)

obs.shape=torch.Size([128, 8, 4, 84, 84])
actions.shape=torch.Size([128, 8])
logprobs.shape=torch.Size([128, 8])
rewards.shape=torch.Size([128, 8])
dones.shape=torch.Size([128, 8])
values.shape=torch.Size([128, 8])
last_episode_rewards.shape=torch.Size([8])
last_episode_lengths.shape=torch.Size([8])
next_obs.shape=torch.Size([8, 4, 84, 84])
next_done.shape=torch.Size([8])


## Train

In [34]:
# @launchit.disable_2
global_step = 0
start_time = time.time()
pbar = tqdm(total=HP.global_steps_count)

while global_step < HP.global_steps_count:
    for w in LS.workers:
        w.rollout(HP.rollout_steps_count, 
                  obs, actions, logprobs, rewards, dones, values, 
                  last_episode_rewards, last_episode_lengths, 
                  next_obs, next_done)

    for w in LS.workers:
        w.get_task_result()

    advantages = torch.zeros_like(rewards).to(CONFIG.cuda_device) # [num_steps, num_envs]
    
    with torch.no_grad():
        next_value = LS.agent.get_value(next_obs).reshape(1, -1)
        lastgaelam = 0
        
        for t in reversed(range(HP.rollout_steps_count)):
            if t == HP.rollout_steps_count - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
                
            delta = rewards[t] + HP.gamma * nextvalues * nextnonterminal - values[t]
            lastgaelam = delta + HP.gamma * HP.gae_lambda * nextnonterminal * lastgaelam
            advantages[t] = lastgaelam
            
        returns = advantages + values

    b_obs = obs.reshape((-1,) + LS.env.observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + LS.env.action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    # Optimizing the policy and value network
    clipfracs = []
    
    for epoch in range(HP.epochs_count):
        for mb_inds in minibatch_data_loader:
            _, newlogprob, entropy, newvalue = LS.agent(b_obs[mb_inds], b_actions.long()[mb_inds])
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                # calculate approx_kl http://joschu.net/blog/kl-approx.html
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > HP.clip_coef).float().mean().item()]

            mb_advantages = b_advantages[mb_inds]
            
            if HP.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            # Policy loss ("pg" stands for policy gradient)
            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - HP.clip_coef, 1 + HP.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            # Value loss
            newvalue = newvalue.view(-1)
            
            if HP.clip_vloss:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -HP.clip_coef,
                    HP.clip_coef,
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            entropy_loss = entropy.mean()
            loss = pg_loss - HP.ent_coef * entropy_loss + v_loss * HP.vf_coef

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(LS.agent.parameters(), HP.max_grad_norm)
            optimizer.step()

        if HP.target_kl is not None and approx_kl > HP.target_kl:
            break

    lr_scheduler.step()
    
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    # TRY NOT TO MODIFY: record rewards for plotting purposes
    summary_writer.add_scalar("charts/sps", int(global_step / (time.time() - start_time)), global_step, is_batched=True) # steps per second
    summary_writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step, is_batched=True)
    summary_writer.add_scalar("charts/episodic_return", last_episode_rewards.mean(), global_step, is_batched=True)
    summary_writer.add_scalar("charts/episodic_length", last_episode_lengths.mean(), global_step, is_batched=True)
    
    summary_writer.add_scalar("losses/value_loss", v_loss.item(), global_step, is_batched=True)
    summary_writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step, is_batched=True)
    summary_writer.add_scalar("losses/entropy", entropy_loss.item(), global_step, is_batched=True)
    summary_writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step, is_batched=True)
    summary_writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step, is_batched=True)
    summary_writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step, is_batched=True)
    summary_writer.add_scalar("losses/explained_variance", explained_var, global_step, is_batched=True)

    if capture_video_manager.should_capture_video(global_step) or (global_step + batch_size >= HP.global_steps_count):
        capture_video_manager.schedule_capture_video(global_step)

    capture_video_manager.upload_captured_video(is_drain=False)
    summary_writer.flush()
    
    pbar.update(min(batch_size, HP.global_steps_count - global_step)) # min is used to not overflow progress bar at the end (global_step could get > HP.global_step_size)
    global_step += batch_size

capture_video_manager.upload_captured_video(is_drain=True)
summary_writer.flush()
pbar.close()

  0%|          | 0/10000 [00:00<?, ?it/s]

## Save

In [ ]:
# @launchit.disable_2
artifact_registry = new_artifact_registry()
lc = HP.launch_component()

with io.BytesIO() as b:
    torch.save(LS.agent.state_dict(), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)

with io.StringIO() as b:
    json.dump(dataclasses.asdict(LS.agent.params), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

# LaunchIt!

## TRAIN

In [36]:
# @launchit.disable
launchit_t0 = time.time()

In [37]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    artifact_registry_obj = new_artifact_registry(is_real=True)
    artifact_registry_obj.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=component_version, expandvars=expandvars, collect_inds=[1], disable_inds=[1])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=2
Creating /home/misha/dev/mine/neurolab/17_rl/17a_ppo_atari_mp_03-launch2.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17a_ppo_atari_mp_03-launch2.ipynb"


Process SpawnProcess-10:
Process SpawnProcess-9:
Process SpawnProcess-8:
Process SpawnProcess-7:
Process SpawnProcess-6:
Traceback (most recent call last):
  File "/home/misha/anaconda3/envs/mine/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/misha/anaconda3/envs/mine/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/misha/dev/mine/neurolab/run/17_rl/17a_ppo_atari_mp_03-launch15.py", line 828, in worker_loop
    task = inp_queue.get(block=True, timeout=task_wait_timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/misha/anaconda3/envs/mine/lib/python3.12/multiprocessing/queues.py", line 113, in get
    if not self._poll(timeout):
           ^^^^^^^^^^^^^^^^^^^
  File "/home/misha/anaconda3/envs/mine/lib/python3.12/multiprocessing/connection.py", line 257, in poll
    return self._poll(timeout)
           ^^^^^^^^^^^^^^^^^^^
  File "/

## Optuna (model selection)

### Templates

In [45]:
# @launchit.disable
# @launchit.collect_3
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP = Hyperparameters()
            HP.random_seed = 42
            assert False
        case _:
            assert False, f'Unsupported {study_serial=}'            

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 1
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")